# 📊 4장 · 그림으로 데이터와 대화하기: 탐색적 분석과 시각화 (개정판 실습 노트북)

『빅데이터분석 with 파이썬』 4장의 **05. 파이썬 실습**과 **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 4-1, 4-2, 4-3, 4-4, 4-5, 4-6 |
| ☆ 선택 | 자습이나 심화 | 4-7, 4-8, 4-9, 4-10, 4-11, 4-12 |

**데이터**
- 따릉이: UCI "Seoul Bike Sharing Demand" 8,760시간(2017년 12월~2018년 11월). GitHub 저장소 `syedsharin/Seoul-Bike-Sharing-Demand-Prediction`에서 바로 읽습니다.
- 미니 프로젝트: 강의 자료 파일 `서울_미세먼지_날씨_시간별_2023_2025.csv`(에어코리아 서울 도시대기 측정소 25곳 평균 + 기상청 ASOS 서울 108, 출처: 한국환경공단·기상청, 공공누리 제1유형). **코랩 왼쪽 파일 창(📁)에 올린 뒤** 미니 프로젝트 셀을 실행하십시오.

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**를 설치합니다. 판다스, 사이킷런, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch04/data/"
for f in [
    "서울_미세먼지_날씨_시간별_2023_2025.csv",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 05. 파이썬 실습: 따릉이 기록을 그림으로

이 절의 실습은 코랩(Colab)에서 위에서부터 차례로 실행합니다. 따릉이 기록은 인터넷 주소에서 바로 읽습니다. 그래프는 맷플롯립(matplotlib)으로 그립니다. 한글은 koreanize-matplotlib로 설정합니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다.

#### [파이썬 실습 4-1 ★] 3장의 규칙으로 표를 만들고 1년을 그리기

시간 단위 기록을 읽어 3장의 정제 규칙을 적용하고, 하루 단위 표를 만드시오. 1년의 하루 대여량을 7일 이동평균과 함께 그리시오.

① 한글 글꼴을 설정하고 데이터를 읽습니다. ② 3장의 규칙 세 가지(운영한 시간만, 습도 0%는 결측, 공식 공휴일)를 적용하고 하루 단위로 묶습니다. ③ 선 그래프를 그리고, 달별 평균과 가장 많고 적은 날을 찾습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob, os
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False  # 음수 기호(-)가 깨지지 않게

URL = ("https://raw.githubusercontent.com/syedsharin/"
       "Seoul-Bike-Sharing-Demand-Prediction/main/SeoulBikeData.csv")
df = pd.read_csv(URL, encoding="latin-1")          # °C 기호 때문에 latin-1
df.columns = ["date", "count", "hour", "temp", "humidity", "wind",
              "visibility", "dew", "solar", "rain", "snow", "season",
              "holiday", "functioning"]
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y")   # 일/월/년

# 3장의 정제 규칙 세 가지를 그대로
h = df[df["functioning"] == "Yes"].copy()          # ① 운영한 시간만
h.loc[h["humidity"] == 0, "humidity"] = np.nan     # ② 습도 0%는 결측
hol = pd.to_datetime(["2017-12-25", "2018-01-01", "2018-02-15", "2018-02-16",
                      "2018-02-17", "2018-03-01", "2018-05-05", "2018-05-07",
                      "2018-05-22", "2018-06-06", "2018-06-13", "2018-08-15",
                      "2018-09-23", "2018-09-24", "2018-09-25", "2018-09-26",
                      "2018-10-03", "2018-10-09"])  # ③ 공식 공휴일 18일
h["dayoff"] = (h["date"].dt.dayofweek >= 5) | h["date"].isin(hol)  # 쉬는 날

day = (h.groupby("date")
         .agg(count=("count", "sum"), temp=("temp", "mean"),
              humidity=("humidity", "mean"), rain=("rain", "sum"),
              hours=("hour", "size"), dayoff=("dayoff", "first"))
         .reset_index())
day = day[day["hours"] == 24].reset_index(drop=True)  # 24시간이 다 있는 날
print("운영한 시간:", len(h), "/ 하루 단위 표:", len(day), "일")
print("평일", (~day["dayoff"]).sum(), "일, 쉬는 날", day["dayoff"].sum(), "일")

이 장은 3장의 정제 규칙을 원본에 그대로 다시 적용해 출발합니다. 3장의 파일(bike_daily_clean.csv, bike_hourly_clean.csv)이 없어도 이 장만 따로 돌릴 수 있게 하기 위해서입니다. 규칙이 같으니 결과도 같습니다. 운영한 8,465시간, 352일, 평일 237일과 쉬는 날 115일입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))           # 도화지 1장, 칸 1개
ax.plot(day["date"], day["count"], color="lightgray", label="하루 대여량")
week = day["count"].rolling(7, center=True).mean() # 앞뒤 7일 평균
ax.plot(day["date"], week, color="tab:red", label="7일 이동평균")
ax.set_ylabel("하루 대여량(대)")
ax.legend()
plt.show()

month = day.groupby(day["date"].dt.month)["count"].mean()
print("1월 하루 평균:", round(month[1]), "대 / 6월:", round(month[6]), "대")
print(day.nlargest(3, "count")[["date", "count", "rain"]].to_string(index=False))
print(day.nsmallest(3, "count")[["date", "count", "rain"]].to_string(index=False))

1월의 하루 평균은 4,839대, 6월은 29,896대입니다. 가장 많은 세 날은 모두 비가 오지 않은 6월의 날입니다. 가장 적은 세 날은 모두 비가 20mm 이상 왔습니다. `rolling(7, center=True)`는 그날을 가운데 두고 앞뒤 7일씩 묶어 평균을 냅니다.

> **[코드 읽기] `fig, ax = plt.subplots( )`**
>
> `plt.subplots( )`는 도화지(fig)와 그 위의 그림 칸(ax)을 한 번에 만듭니다. 그다음에는 `ax.plot( )`, `ax.set_ylabel( )`처럼 칸에 대고 그립니다. `plt.subplots(1, 2)`처럼 줄과 칸 수를 주면 칸이 여러 개 생기고, `axes[0]`, `axes[1]`로 하나씩 고릅니다. `figsize=(9, 3.5)`는 도화지의 가로와 세로(인치)입니다.

#### [파이썬 실습 4-2 ★] 한 시간 대여량의 히스토그램: 보통 눈금과 로그 눈금

한 시간 대여량의 히스토그램을 보통 눈금과 로그 눈금으로 나란히 그리시오. 평균과 중앙값을 구하고, 100대 미만인 시간을 살펴보시오.

① 칸 두 개짜리 도화지를 만듭니다. ② 왼쪽은 같은 폭의 칸 30개로, 오른쪽은 같은 배수로 늘어나는 칸 30개로 그립니다. ③ 100대 미만인 시간의 시각과 비를 셉니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))  # 칸 2개를 가로로
axes[0].hist(h["count"], bins=30, color="tab:blue", edgecolor="white")
axes[0].set_title("보통 눈금")
log_bins = np.logspace(np.log10(2), np.log10(3556), 31)  # 같은 배수로 늘어나는 경계
axes[1].hist(h["count"], bins=log_bins, color="tab:blue", edgecolor="white")
axes[1].set_xscale("log")                          # 가로축을 로그 눈금으로
axes[1].set_title("로그 눈금")
for ax in axes:
    ax.set_xlabel("한 시간 대여량(대)")
axes[0].set_ylabel("시간 수")
plt.show()

print(f"평균 {h['count'].mean():.1f}대, 중앙값 {h['count'].median():.0f}대")
small = h[h["count"] < 100]                        # 100대 미만인 시간
print(f"100대 미만: {len(small)}시간 ({len(small) / len(h):.1%})")
print(f"그 가운데 새벽 3~6시: {small['hour'].between(3, 6).mean():.1%}")
print(f"비 온 시간의 비율: 100대 미만에서 {(small['rain'] > 0).mean():.1%},",
      f"전체에서 {(h['rain'] > 0).mean():.1%}")

평균(729.2대)이 중앙값(542대)보다 큽니다. 오른쪽으로 꼬리가 긴 분포에서 흔히 보이는 모습입니다. 100대 미만인 916시간 가운데 절반 남짓이 새벽 3~6시였습니다. 이 시간들의 33.2%에 비가 왔는데, 전체에서는 6.1%였습니다. `np.logspace( )`는 2에서 3,556까지를 같은 "배수"로 나누는 칸 경계를 만듭니다.

#### [파이썬 실습 4-3 ★] 달마다 상자 그림

하루 대여량의 상자 그림을 달마다 그리시오. 1년 전체의 사분위수, 상자가 가장 긴 달, 수염 밖으로 떨어진 날을 구하시오.

① `boxplot(column=…, by="month")`로 달마다 상자를 그립니다. ② `quantile( )`로 사분위수와 달별 IQR을 구합니다. ③ 달마다 울타리 밖의 날을 골라 비가 왔는지 셉니다.

In [ ]:
day["month"] = day["date"].dt.month
fig, ax = plt.subplots(figsize=(9, 3.6))
day.boxplot(column="count", by="month", ax=ax, grid=False)
ax.set_title("")
fig.suptitle("")                                   # 판다스가 붙이는 제목 지우기
ax.set_xlabel("월")
ax.set_ylabel("하루 대여량(대)")
plt.show()

q1, q3 = day["count"].quantile([0.25, 0.75])       # 아래·위 사분위수
print("1년 전체: Q1", round(q1), "/ 중앙값", round(day["count"].median()),
      "/ Q3", round(q3), "/ IQR", round(q3 - q1))
iqr = day.groupby("month")["count"].quantile(0.75) - day.groupby("month")["count"].quantile(0.25)
print("상자가 가장 긴 달:", iqr.idxmax(), "월, IQR", round(iqr.max()))

def far(g):                                         # 수염 밖으로 떨어진 날
    a, b = g["count"].quantile([0.25, 0.75])
    return g[(g["count"] < a - 1.5 * (b - a)) | (g["count"] > b + 1.5 * (b - a))]
odd = day.groupby("month", group_keys=False).apply(far)
print("수염 밖의 날:", len(odd), "일, 그 가운데 비가 5mm 넘게 온 날:", (odd["rain"] > 5).sum(), "일")

4월의 상자가 가장 깁니다(IQR 11,720대). 수염 밖의 14일은 모두 비가 5mm 넘게 왔습니다. `far( )`는 3장에서 배운 울타리를 달마다 따로 적용하는 함수입니다. `groupby("month").apply(far)`는 달마다 표를 잘라 이 함수에 넣고, 결과를 다시 이어 붙입니다.

#### [파이썬 실습 4-4 ★] 평일과 쉬는 날의 하루 모양

시각별 평균 대여량을 평일과 쉬는 날로 나누어 선 그래프로 그리시오. 섞은 평균과 견주시오.

① 시각과 쉬는 날 여부로 묶어 평균을 냅니다. ② `unstack( )`으로 쉬는 날 여부를 열로 펼칩니다. ③ 판다스의 `plot( )`으로 두 선을 그립니다.

In [ ]:
prof = h.groupby(["hour", "dayoff"])["count"].mean().unstack()
prof.columns = ["평일", "쉬는 날"]
ax = prof.plot(figsize=(9, 3.6), marker="o", ms=4, color=["tab:blue", "tab:red"])
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("시각(시)")
ax.set_ylabel("시간당 평균 대여량(대)")
plt.show()

mixed = h.groupby("hour")["count"].mean()          # 평일과 쉬는 날을 섞은 평균
print("섞은 평균:", mixed[[4, 8, 18]].round().astype(int).to_dict())
print(prof.loc[[4, 8, 12, 17, 18]].round().astype(int))

섞은 평균의 8시는 1,050대입니다. 평일은 1,357대, 쉬는 날은 424대입니다. 4시만은 쉬는 날(175대)이 평일(119대)보다 많습니다. 쉬는 날 새벽에 귀가하는 사람들일까요? 이것도 그림이 낳은 질문입니다.

#### [파이썬 실습 4-5 ★] 시각 × 달 히트맵

시각과 달마다 시간당 평균 대여량을 표로 만들고 히트맵으로 그리시오. 18시 줄의 값, 여름 18시의 평균기온, 달별 강수량을 구하시오.

① `pivot_table( )`로 24줄 × 12칸 표를 만듭니다. ② `imshow( )`로 숫자를 색으로 바꿉니다. ③ 18시 줄과 기온, 강수량을 뽑아 봅니다.

In [ ]:
h["month"] = h["date"].dt.month
table = h.pivot_table(index="hour", columns="month",
                      values="count", aggfunc="mean")   # 24줄 × 12칸
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(table, cmap="Reds", aspect="auto")  # 숫자를 색으로
ax.set_xticks(range(12), table.columns)
ax.set_yticks(range(0, 24, 2), range(0, 24, 2))
ax.set_xlabel("월")
ax.set_ylabel("시각(시)")
fig.colorbar(im, label="시간당 평균 대여량(대)")
plt.show()

print("18시 줄:", table.loc[18].round().astype(int).to_dict())
t18 = h[h["hour"] == 18].groupby("month")["temp"].mean()
print("18시 평균기온:", t18[[6, 7, 8, 9]].round(1).to_dict())
print("달마다 강수량 합:", h.groupby("month")["rain"].sum()[[5, 6, 7, 8, 9]].round(1).to_dict())

18시 줄은 6월 2,494대에서 8월 1,837대로 내려갔다가 9월에 2,319대로 돌아옵니다. 7~8월 18시의 평균기온은 30도를 넘었습니다. 7~8월의 강수량도 9월보다 훨씬 많았습니다.

> **[코드 읽기] `pivot_table( )`**
>
> `pivot_table(index="hour", columns="month", values="count", aggfunc="mean")`은 "hour를 줄로, month를 칸으로 놓고, 칸마다 count의 평균을 넣어라"라는 뜻입니다. 3장의 `groupby( )`로 두 열을 묶은 뒤 펼친 것과 같은 결과를 한 번에 냅니다.

#### [파이썬 실습 4-6 ★] 기온 구간, 산점도, 상관계수

하루 평균기온을 4도 구간으로 나누어 구간별 날수와 평균을 구하시오. 기온과 습도의 산점도를 그리고, 상관계수를 구하시오.

① `pd.cut( )`으로 기온 구간을 만들고, 구간에 들지 못한 날이 있는지 셉니다. ② 구간별 날수와 평균을 구해 산점도 위에 선으로 겹칩니다. ③ 상관계수를 구하고 습도가 높은 날과 낮은 날을 들여다봅니다.

In [ ]:
bands = pd.cut(day["temp"], bins=range(-16, 40, 4))   # 4도씩: -16도 초과 ~ 36도 이하
print("구간에 들지 못한 날:", bands.isna().sum(), "일")
by = day.groupby(bands, observed=True).agg(날수=("count", "size"),
                                            평균기온=("temp", "mean"),
                                            평균대여=("count", "mean"))
print(by.tail(5).round(1).to_string())
hot = day["temp"] > 28                             # 28도 넘는 날을 한데 모으면
print("28도 넘는 날:", hot.sum(), "일, 하루 평균", round(day.loc[hot, "count"].mean()), "대")

rainy = day["rain"] >= 1                           # 하루 1mm 이상 비 온 날
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
axes[0].scatter(day["temp"], day["count"], s=10, alpha=0.5, color="tab:blue")
axes[0].plot(by["평균기온"], by["평균대여"], color="tab:red", marker="o")
axes[0].set_xlabel("하루 평균기온(°C)")
axes[0].set_ylabel("하루 대여량(대)")
axes[1].scatter(day.loc[~rainy, "humidity"], day.loc[~rainy, "count"],
                s=10, alpha=0.5, color="tab:blue", label="비 안 온 날")
axes[1].scatter(day.loc[rainy, "humidity"], day.loc[rainy, "count"],
                s=12, alpha=0.8, color="tab:red", label="비 온 날")
axes[1].set_xlabel("하루 평균습도(%)")
axes[1].legend()
plt.show()

구간에 들지 못한 날은 0일입니다. 24~28도 구간의 평균이 가장 높고, 32~36도 구간은 4일뿐입니다. 28도를 넘는 36일을 모으면 하루 평균 22,579대입니다.

> **[코드 읽기] `pd.cut( )`**
>
> `pd.cut(값, bins=range(-16, 40, 4))`는 −16, −12, …, 36을 경계로 값을 구간에 나눕니다. `(24, 28]`은 "24 초과 28 이하"입니다. 경계 밖의 값은 NaN이 되어 조용히 빠집니다. 그래서 `isna().sum( )`으로 빠진 값이 없는지 꼭 셉니다. 경계를 36까지 두지 않으면 가장 더운 4일이 그래프에서 사라집니다.

In [ ]:
r_temp = day["count"].corr(day["temp"])
r_hum = day["count"].corr(day["humidity"])
r_rain = day["count"].corr(day["rain"])
print(f"상관계수  기온 {r_temp:.2f}  습도 {r_hum:.2f}  강수량 {r_rain:.2f}")
wet = day[day["humidity"] > 80]
print("습도 80% 넘는 날:", len(wet), "일, 그 가운데 비 온 날:", (wet["rain"] >= 1).sum(), "일")
print("습도 40% 이하인 날의 평균기온:", round(day.loc[day["humidity"] <= 40, "temp"].mean(), 1), "도")

기온의 상관계수는 0.75, 습도는 0.04, 강수량은 −0.24입니다. 습도가 80%를 넘은 35일은 모두 비 온 날이었고, 습도 40% 이하인 날의 평균기온은 0.7도였습니다.

#### [파이썬 실습 4-7 ☆] 정직한 축과 잘린 축

평일과 쉬는 날의 하루 평균을 0부터 그린 막대와 16,000부터 자른 막대로 나란히 그리고, 거짓말 계수를 구하시오.

In [ ]:
means = day.groupby("dayoff")["count"].mean()
means.index = ["평일", "쉬는 날"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].bar(means.index, means.values, color=["tab:blue", "tab:red"])
axes[0].set_title("정직한 축 (0부터)")
axes[1].bar(means.index, means.values, color=["tab:blue", "tab:red"])
axes[1].set_ylim(16000, 18500)                     # 세로축을 16,000부터
axes[1].set_title("잘린 축 (16,000부터)")
plt.show()

real = means["평일"] / means["쉬는 날"]
shown = (means["평일"] - 16000) / (means["쉬는 날"] - 16000)
print(means.round().astype(int).to_dict())
print(f"실제 비 {real:.3f}, 잘린 축의 막대 높이 비 {shown:.1f}")
print(f"거짓말 계수 {(shown - 1) / (real - 1):.0f}")
weekend = day["date"].dt.dayofweek >= 5            # 토·일만 주말로 (공휴일은 평일 쪽)
by_we = day.groupby(weekend)["count"].agg(["size", "mean"]).round().astype(int)
by_we.index = ["평일(공휴일 포함)", "주말(토·일)"]
print(by_we.to_string())
print("차이:", by_we["mean"].iloc[0] - by_we["mean"].iloc[1], "대")

실제 비는 1.091인데 잘린 축의 막대 높이 비는 4.1입니다. 거짓말 계수는 (4.1 − 1) ÷ (1.091 − 1) ≈ 34입니다.

마지막 표는 공휴일을 평일 쪽에 두고 토·일만 주말로 나눈 것입니다. 평일 251일은 하루 평균 17,955대, 주말 101일은 16,333대로 차이가 1,622대입니다. 5장은 이 나눔으로 평일과 주말을 견줍니다. 공휴일을 어느 쪽에 넣느냐에 따라 같은 352일의 차이가 1,499대도 되고 1,622대도 됩니다.

#### [파이썬 실습 4-8 ☆] 계절마다 나누어 그리기

계절마다 평일과 쉬는 날의 시각별 평균을 그리되, 네 그림의 축을 같게 맞추시오.

In [ ]:
seasons = ["Winter", "Spring", "Summer", "Autumn"]
names = ["겨울", "봄", "여름", "가을"]
fig, axes = plt.subplots(2, 2, figsize=(10, 6.4), sharex=True, sharey=True)  # 축을 함께
for ax, s, name in zip(axes.flat, seasons, names):
    part = h[h["season"] == s]
    p = part.groupby(["hour", "dayoff"])["count"].mean().unstack()
    ax.plot(p.index, p[False], color="tab:blue", label="평일")
    ax.plot(p.index, p[True], color="tab:red", label="쉬는 날")
    ax.set_title(name)
    ax.set_xticks(range(0, 24, 6))
axes[1, 0].set_xlabel("시각(시)")
axes[1, 0].set_ylabel("시간당 평균 대여량(대)")
axes[0, 0].legend()
plt.show()
summer = h[h["season"] == "Summer"]
print("여름 쉬는 날 가장 많은 시각:", summer[summer["dayoff"]].groupby("hour")["count"].mean().idxmax(), "시")

`sharex=True, sharey=True`로 네 칸의 축을 맞추었습니다. 축이 칸마다 다르면 겨울의 작은 봉우리가 여름만큼 커 보입니다. 여름 쉬는 날에는 20시에 가장 많이 빌렸습니다.

#### [파이썬 실습 4-9 ☆] 앤스컴 4종 세트

앤스컴의 네 데이터에서 평균, 분산, 상관계수, 직선을 구하고 네 산점도를 그리시오.

In [ ]:
x = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
ans = {"I": (x, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
       "II": (x, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
       "III": (x, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
       "IV": ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
              [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])}
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharex=True, sharey=True)
for ax, (name, (xs, ys)) in zip(axes, ans.items()):
    xs, ys = np.array(xs), np.array(ys)
    b1, b0 = np.polyfit(xs, ys, 1)                 # 가장 잘 맞는 직선
    r = np.corrcoef(xs, ys)[0, 1]
    print(f"{name:3s} x평균 {xs.mean():.1f}  y평균 {ys.mean():.2f}  y분산 {ys.var(ddof=1):.2f}"
          f"  상관 {r:.3f}  직선 y = {b0:.2f} + {b1:.3f}x")
    ax.scatter(xs, ys, color="tab:blue")
    ax.plot([3, 20], [b0 + b1 * 3, b0 + b1 * 20], color="tab:red")
    ax.set_title(name)
plt.show()

네 데이터의 요약 숫자는 소수 둘째 자리까지 거의 같습니다. 그러나 산점도는 넷이 다 다릅니다.

#### [파이썬 실습 4-10 ☆] 나이팅게일의 로즈 다이어그램

크림 전쟁 영국군의 달별 사망률을 극좌표 막대로 그리시오. 쐐기의 넓이가 값에 비례하도록 반지름에 제곱근을 쓰시오.

In [ ]:
months_kr = ["4월", "5월", "6월", "7월", "8월", "9월", "10월", "11월",
             "12월", "1월", "2월", "3월"]
disease = [1.4, 6.2, 4.7, 150, 328.5, 312.2, 197, 340.6, 631.5, 1022.8, 822.8, 480.3,
           177.5, 171.8, 247.6, 107.5, 129.9, 47.5, 32.8, 56.4, 25.3, 11.4, 6.6, 3.9]
wounds = [0, 0, 0, 0, 0.4, 32.1, 51.7, 115.8, 41.7, 30.7, 16.3, 12.8,
          17.9, 16.6, 64.5, 37.7, 44.1, 69.4, 13.6, 10.5, 5, 0.5, 0, 0]
other = [7, 4.6, 2.5, 9.6, 11.9, 27.7, 50.1, 42.8, 48, 120, 140.1, 68.6,
         21.2, 12.5, 9.6, 9.3, 6.7, 5, 4.6, 10.1, 7.8, 13, 5.2, 9.1]
theta = np.arange(12) * 2 * np.pi / 12             # 한 달 = 30도
fig, axes = plt.subplots(1, 2, figsize=(11, 5.8), subplot_kw={"projection": "polar"})
titles = ["1854년 4월 ~ 1855년 3월", "1855년 4월 ~ 1856년 3월"]
for k, ax in enumerate(axes):
    part = slice(12 * k, 12 * k + 12)
    for vals, color, name in [(disease, "#C8234B", "막을 수 있던 병"),
                              (other, "#555555", "그 밖의 원인"),
                              (wounds, "#7FB3D5", "전투 부상")]:
        ax.bar(theta, np.sqrt(vals[part]), width=2 * np.pi / 12,   # 넓이가 값에 비례하게
               color=color, alpha=0.8, edgecolor="white", label=name)
    ax.set_theta_zero_location("N")                # 4월을 12시 방향에
    ax.set_theta_direction(-1)                     # 시계 방향으로
    ax.set_xticks(theta, months_kr)
    ax.set_yticklabels([])
    ax.set_ylim(0, np.sqrt(1022.8))                # 두 그림의 크기를 같게
    ax.set_title(titles[k], pad=18)
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=3)
plt.show()
n_disease, n_wounds, n_other = 14476, 1758, 1748   # 24개월 사망자 수 합계
total = n_disease + n_wounds + n_other
print(f"병 {n_disease:,}명 ({n_disease / total:.1%}), 부상 {n_wounds:,}명, "
      f"그 밖 {n_other:,}명 / 병은 부상의 {n_disease / n_wounds:.1f}배")

24개월 사망자 17,982명 가운데 병이 80.5%였습니다. 병으로 죽은 사람이 부상으로 죽은 사람의 8.2배입니다.

#### [파이썬 실습 4-11 ☆] 64차원 사진을 2차원 지도로

손글씨 숫자 사진 1,797장을 PCA와 t-SNE로 2차원에 펼치고, 가장 가까운 이웃이 같은 숫자인 비율을 견주시오.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

digits = load_digits()
X, y = digits.data, digits.target
print(X.shape)                                     # (장 수, 칸 수)
maps = {"PCA": PCA(n_components=2).fit_transform(X),
        "t-SNE": TSNE(n_components=2, init="pca", random_state=0).fit_transform(X)}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (name, xy) in zip(axes, maps.items()):
    ax.scatter(xy[:, 0], xy[:, 1], c=y, cmap="tab10", s=6)
    for d in range(10):                            # 숫자마다 가운데에 이름표
        cx, cy = np.median(xy[y == d], axis=0)
        ax.text(cx, cy, str(d), fontsize=14, weight="bold", ha="center", va="center")
    nn = NearestNeighbors(n_neighbors=2).fit(xy)
    idx = nn.kneighbors(xy, return_distance=False)[:, 1]   # 자기 다음으로 가까운 점
    agree = (y[idx] == y).mean()
    ax.set_title(f"{name}: 이웃이 같은 숫자 {agree:.1%}")
    ax.set_xticks([]); ax.set_yticks([])
    print(name, round(agree, 3))
plt.show()

가장 가까운 이웃이 같은 숫자인 비율은 PCA 58.7%, t-SNE 98.8%입니다. t-SNE는 실행할 때마다 지도가 조금씩 달라서 `random_state=0`으로 고정했습니다.

#### [파이썬 실습 4-12 ☆] AI가 그린 그래프 점검하기

운영하지 않은 시간을 거르지 않은 평균과 거른 평균을 견주시오. 10월의 하루 대여량을 막대로 그려 0인 날을 찾으시오.

In [ ]:
naive = df.groupby("hour")["count"].mean()         # 운영 여부를 거르지 않음
clean = h.groupby("hour")["count"].mean()          # 운영한 시간만
print("오후 6시 평균:", round(naive[18], 1), "→", round(clean[18], 1))
daily_all = df.groupby("date")["count"].sum()      # 운영 안 한 날도 하루로 셈
oct_all = daily_all[daily_all.index.month == 10]
oct_clean = day.loc[day["date"].dt.month == 10, "count"]
print(f"10월 하루 평균: {oct_all.mean():,.0f}대 ({len(oct_all)}일) → "
      f"{oct_clean.mean():,.0f}대 ({len(oct_clean)}일)")
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(oct_all.index.day, oct_all.values, color="tab:blue")
ax.set_xlabel("2018년 10월(일)")
ax.set_ylabel("하루 대여량(대)")
plt.show()
print("10월에 0대인 날:", list(oct_all[oct_all == 0].index.day))

거르지 않으면 18시 평균이 1,502.9대, 거르면 1,554.0대입니다. 10월의 하루 평균은 20,990대에서 23,505대로 바뀝니다. 그래프만 보면 두 숫자 모두 그럴듯합니다.

> **핵심 3줄**
>
> 그래프는 질문마다 고릅니다. 분포는 히스토그램과 상자 그림으로, 하루와 한 해의 모양은 선과 히트맵으로, 두 변수의 관계는 산점도와 상관계수로 봅니다.
>
> 그림은 빠진 것을 알려 주지 않습니다. 구간을 나누거나 0을 거를 때는 빠진 값이 없는지 코드로 먼저 확인합니다.
>
> 4도 구간은 한 날도 빠뜨리지 않았고, 기온과 하루 대여량의 상관계수는 0.75, 꼭대기는 24~28도(하루 평균 26,630대)였습니다.

## [미니 프로젝트 4] 서울의 미세먼지는 언제 나쁠까

**질문.** "봄에는 미세먼지가 심하다", "출퇴근 시간에는 차가 많아 공기가 나쁘다"는 말을 자주 듣습니다. 정말 그럴까요? 서울의 3년 치 기록을 그림으로 그려 보며, 미세먼지가 언제 나쁜지 질문을 만들어 보십시오. 이 프로젝트는 모델을 만들지 않습니다. 그리고, 묻고, 아직 모르는 것을 적습니다.

**데이터.** 강의 자료로 나누어 주는 「서울_미세먼지_날씨_시간별_2023_2025.csv」(1.5MB)입니다. 기간은 2023년 1월 1일 1시부터 2026년 1월 1일 0시까지입니다. 한 행이 한 시각이고, 26,304행 10열입니다. PM10과 PM2.5는 에어코리아 확정자료에서 서울 도시대기 측정소 25곳의 시간별 값을 평균했습니다. 기온, 강수량, 풍속, 습도와 황사 기록은 기상청 서울 관측소(ASOS 108)의 시간 자료입니다. 둘을 "일시"로 붙였습니다.


**1단계: 데이터 신원 확인**([파이썬 실습 4-1] 응용). 파일을 읽고 열마다 빈칸을 세시오. 측정일시를 날짜로 바꾸어 보고, 측정소가 25곳보다 적은 시각과 강수량의 빈칸을 살펴보시오.

In [ ]:
PATH = "서울_미세먼지_날씨_시간별_2023_2025.csv"   # 코랩 왼쪽 파일 창에 올린 파일
air = pd.read_csv(PATH)
print(air.shape)
print(air.head(3).to_string())
print(air.isna().sum().to_dict())                  # 열마다 빈칸 수

try:
    pd.to_datetime(air["측정일시"].astype(str), format="%Y%m%d%H")
except ValueError as e:
    print("날짜로 바꾸지 못함:", str(e)[:58])
print("측정일시의 시:", (air["측정일시"] % 100).min(), "~", (air["측정일시"] % 100).max())
air["일시"] = pd.to_datetime(air["일시"])          # 24시를 다음 날 0시로 바꿔 둔 열
print("측정소가 25곳보다 적은 시각:", (air["측정소수"] < 25).sum(),
      "/ 가장 적을 때", air["측정소수"].min(), "곳")
print("비가 온 시각:", (air["강수량"] > 0).sum(), "/ 강수량이 빈칸인 시각:", air["강수량"].isna().sum())

측정일시를 그대로 날짜로 바꾸면 오류가 납니다. 에어코리아는 하루를 1시부터 24시까지 적기 때문입니다. 24시는 다음 날 0시입니다. 그래서 파일에 "일시" 열을 따로 두었습니다. 측정소가 25곳보다 적은 시각이 10,845시각이고, 가장 적을 때는 17곳이었습니다. 강수량의 빈칸 23,110시각은 대부분 비가 오지 않은 시각입니다. 기상청은 비가 오지 않으면 칸을 비워 둡니다.

**2단계: 분포와 기준 숫자**([파이썬 실습 4-2] 응용). PM10과 PM2.5를 요약하고, PM2.5가 "나쁨"(36 이상)인 시각의 비율을 구하시오. PM2.5의 히스토그램을 로그 눈금으로 그리시오.

In [ ]:
print(air[["PM10", "PM25"]].describe().round(1).to_string())
print(f"PM2.5 '나쁨'(36 이상) 시각의 비율: {(air['PM25'] >= 36).mean():.1%}")
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(air["PM25"], bins=np.logspace(0, np.log10(140), 31), color="tab:blue", edgecolor="white")
ax.set_xscale("log")
ax.axvline(36, color="tab:red", linestyle="--")    # '나쁨'이 시작되는 값
ax.set_xlabel("PM2.5 농도(로그 눈금)")
ax.set_ylabel("시각 수")
plt.show()

PM2.5의 평균은 18.6, 중앙값은 15.4입니다. 단위는 ㎍/㎥입니다. "나쁨"인 시각은 9.3%입니다. 1장에서는 2026년 4월까지 넣어 평균 19.2, "나쁨" 10.6%였습니다. 기간이 달라 숫자가 조금 다릅니다. 이 숫자가 3단계에서 견줄 기준입니다.

**3단계: 시각, 달, 요일로 나누기**([파이썬 실습 4-4], [4-5] 응용). PM2.5 평균을 시각별, 달별, 요일별로 그리시오.

In [ ]:
air["시"] = air["일시"].dt.hour
air["월"] = air["일시"].dt.month
air["요일"] = air["일시"].dt.dayofweek               # 0=월 … 6=일
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
air.groupby("시")["PM25"].mean().plot(ax=axes[0], marker="o", color="tab:blue")
air.groupby("월")["PM25"].mean().plot(ax=axes[1], marker="o", color="tab:red")
air.groupby("요일")["PM25"].mean().plot.bar(ax=axes[2], color="tab:gray")
axes[0].set_ylabel("PM2.5 평균 농도")
axes[1].set_xticks(range(1, 13))
axes[2].set_xticks(range(7), list("월화수목금토일"), rotation=0)
for ax, name in zip(axes, ["시각", "월", "요일"]):
    ax.set_xlabel(name)
plt.show()
by_h = air.groupby("시")["PM25"].mean()
by_m = air.groupby("월")["PM25"].mean()
print("시각별 평균:", round(by_h.min(), 1), "~", round(by_h.max(), 1))
print("가장 높은 달:", by_m.idxmax(), "월", round(by_m.max(), 1), "/ 가장 낮은 달:", by_m.idxmin(), "월", round(by_m.min(), 1))

시각별 평균은 17.8에서 19.6 사이로 거의 평평합니다. 달별로는 3월이 27.3으로 가장 높고, 9월이 9.9로 가장 낮습니다. 요일별로도 차이가 거의 없습니다. 출퇴근 시간의 차보다 계절이 더 크게 움직이는 것처럼 보입니다.

**4단계: 튀는 날 들여다보기**([파이썬 실습 4-3] 응용). PM10이 가장 높은 5일을 찾고, 황사 기록과 견주시오. 황사인 시각과 아닌 시각의 PM2.5 ÷ PM10을 견주시오.

In [ ]:
air["날짜"] = air["일시"].dt.normalize()
daily = air.groupby("날짜").agg(PM10=("PM10", "mean"), PM25=("PM25", "mean"), 황사=("황사", "max"))
print(daily.nlargest(5, "PM10").round(1).to_string())
print("황사가 관측된 날:", daily["황사"].sum(), "일 /", air["황사"].sum(), "시간")
top = air.loc[air["PM10"].idxmax()]
print("PM10이 가장 높은 시각:", top["일시"], top["PM10"])
ratio = air["PM25"] / air["PM10"]                  # 미세먼지 가운데 초미세먼지의 몫
print("PM2.5 ÷ PM10 중앙값  황사 시각", round(ratio[air["황사"] == 1].median(), 2),
      "/ 그 밖의 시각", round(ratio[air["황사"] == 0].median(), 2))
dust = air["황사"] == 1
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.scatter(air.loc[~dust, "PM10"], air.loc[~dust, "PM25"], s=3, alpha=0.3, color="tab:blue", label="그 밖의 시각")
ax.scatter(air.loc[dust, "PM10"], air.loc[dust, "PM25"], s=8, alpha=0.8, color="tab:red", label="황사 관측")
ax.set_xlabel("PM10 농도")
ax.set_ylabel("PM2.5 농도")
ax.legend()
plt.show()

PM10이 가장 높은 5일은 모두 황사가 관측된 날이었습니다. 1위인 2023년 4월 12일은 PM10이 266.6인데 PM2.5는 30.1입니다. 황사인 시각의 PM2.5 ÷ PM10 중앙값은 0.23이고, 그 밖의 시각은 0.56입니다. 황사는 굵은 먼지가 많아서 PM10만 크게 올립니다.

**5단계: 한계 쓰기.** 이 그림들이 아직 답하지 못한 질문을 [표 4-3]처럼 표로 쓰시오. 예를 들어 서울 평균 한 줄이 우리 동네의 공기와 같은지 생각해 보십시오. 3년이 계절의 모양을 말하기에 충분한지도 따져 보십시오.

**제출물.** 그림 2장(달별 PM2.5 평균, PM10과 PM2.5의 산점도), 표 1개(PM10이 높은 날과 황사 여부), 결론 5문장 이내.

**[표 4-4] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 24시 표기, 측정소 수, 강수량 빈칸의 뜻을 모두 밝힘 | 빈칸만 셈 | 점검 없이 바로 그림 |
| 기준 숫자(20%) | 평균, 중앙값, "나쁨" 비율을 구하고 견줄 기준으로 씀 | 요약 숫자만 나열함 | 기준 없이 그림만 그림 |
| 그림과 나누기(30%) | 질문에 맞는 그래프로 시각, 달, 요일, 황사를 나누어 그림 | 그림은 맞으나 축이나 단위가 빠짐 | 코드가 돌지 않거나 그림이 틀림 |
| 해석과 한계(30%) | 그림이 낳은 질문과 아직 모르는 것을 표로 씀 | 한계를 한두 가지 씀 | 그림을 결론처럼 씀 |

**교수용 정답 예시.** 핵심 숫자는 이렇습니다. 26,304행 10열, 측정일시는 1~24시로 적혀 있고, 측정소가 25곳보다 적은 시각이 10,845개(가장 적을 때 17곳), 강수량 빈칸 23,110개는 대부분 비가 오지 않은 시각입니다. PM2.5 평균 18.6, 중앙값 15.4, "나쁨" 9.3%. 시각별 평균은 17.8~19.6으로 평평하고, 달별로는 3월 27.3, 9월 9.9입니다. PM10 상위 5일은 모두 황사 날이고, PM2.5 ÷ PM10 중앙값은 황사 시각 0.23, 그 밖 0.56입니다. 결론의 예: "서울의 초미세먼지는 하루 중 시각보다 계절에 따라 크게 달라진다. 봄에 높고 가을에 낮다. 봄의 높은 PM10은 황사의 굵은 먼지 몫이 크다. 다만 서울 평균 한 줄이라 동네마다의 차이와 원인은 이 그림으로 알 수 없다."

> **[더 알아보기] 원자료를 직접 받아 파일 만들기**
>
> 강의 자료 파일은 원자료 두 가지를 붙여 만들었습니다. 미세먼지는 에어코리아 누리집(airkorea.or.kr)의 최종확정 측정자료에서 연도별 압축 파일로 받습니다. 압축을 풀면 달마다 엑셀 파일(예: 2023년 1월.xlsx)이 들어 있습니다. 날씨는 기상자료개방포털(data.kma.go.kr)의 종관기상관측(ASOS)에서 받습니다. 서울(108)의 시간 자료를 CSV로 고릅니다. 둘 다 회원 가입이 필요하고, 공공누리 조건에 따라 출처를 밝혀 씁니다.
>
> 받은 엑셀 파일들을 같은 폴더에 두고 아래 코드(MP-0)를 실행해 보십시오. 서울 도시대기 측정소의 시간별 평균이 만들어집니다. 파일 하나에 1분쯤 걸립니다. 24시를 다음 날 0시로 바꾸는 줄이 핵심입니다. 날씨 자료는 "일시" 열을 기준으로 `merge( )`로 붙입니다. 원자료를 받아 붙일 때는 이런 숨은 약속(24시 표기, 빈칸의 뜻)을 먼저 확인하십시오. 5장 미니 프로젝트에서 쓸 지하철 자료에도 이런 함정이 있습니다. 원자료의 2024년 4월 13~14시 승차 칸이 다른 달의 여덟 배쯤으로 적혀 있습니다.

In [ ]:
# ☆ 에어코리아·기상청에서 받은 원자료로 강의 자료 파일 만들기 (강의 자료의 파일을 쓰면 건너뜁니다)
import glob
AIR = sorted(glob.glob("20??년 *월.xlsx"))         # 에어코리아 확정자료, 한 달에 한 파일
if not AIR:
    print("받은 원자료가 없습니다. 강의 자료의 파일로 1단계를 하십시오.")
else:
    parts = []
    for f in AIR:                                  # 파일 하나에 1분쯤 걸립니다
        t = pd.read_excel(f)
        t = t[t["지역"].str.startswith("서울") & (t["망"] == "도시대기")]
        parts.append(t[["측정소명", "측정일시", "PM10", "PM25"]])
    air = pd.concat(parts, ignore_index=True)
    mp = (air.groupby("측정일시")
             .agg(PM10=("PM10", "mean"), PM25=("PM25", "mean"),
                  측정소수=("PM25", "count")).round(1).reset_index())
    # 측정일시의 시는 01~24: 24시를 다음 날 0시로 바꾼다
    d = pd.to_datetime((mp["측정일시"] // 100).astype(str), format="%Y%m%d")
    mp["일시"] = d + pd.to_timedelta(mp["측정일시"] % 100, unit="h")
    print(len(AIR), "개 파일 →", mp.shape, mp["일시"].min(), "~", mp["일시"].max())

## ✏️ [연습문제 4-5] 쉬는 날의 꼭대기 시각

[파이썬 실습 4-5]의 히트맵을 쉬는 날만으로 다시 만들고, 달마다 가장 많이 빌린 시각을 구하시오. 여름에 꼭대기 시각이 어떻게 바뀌는지 쓰시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
off = h[h["dayoff"]]
t_off = off.pivot_table(index="hour", columns="month", values="count", aggfunc="mean")
print("쉬는 날, 달마다 가장 많이 빌린 시각:", t_off.idxmax().to_dict())

## ✏️ [연습문제 4-6] 비 오는 시각의 미세먼지

미니 프로젝트 파일에서 비가 온 시각(강수량 > 0)과 강수량이 빈칸인 시각의 PM2.5 평균을 견주시오. 이 차이를 "비가 미세먼지를 씻어 낸다"는 증거로 쓸 수 있는지 쓰시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
rain_h = air[air["강수량"] > 0]["PM25"].mean()
dry_h = air[air["강수량"].isna()]["PM25"].mean()
print("비 온 시각", round(rain_h, 1), "/ 강수량 빈칸 시각", round(dry_h, 1))